In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Seed fisso: risultati riproducibili
random.seed(42)
np.random.seed(42)

# I nostri clienti finti
clienti = [
    "Rossi Costruzioni SRL", "Bianchi Alimentari SPA", "Verdi Logistica SRL",
    "Toscana Impianti SNC", "Meccanica Fiorentina SRL", "Studio Neri & Associati",
    "Elettrodomestici Gallo SRL", "Tessuti Lucchesi SPA", "AgriToscana SOC COOP",
    "InfoService SRL"
]

# Generiamo 60 fatture nel primo semestre 2026
fatture = []
for i in range(1, 61):
    data_emissione = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 150))
    fatture.append({
        "numero_fattura": f"FT-2026-{i:03d}",
        "cliente": random.choice(clienti),
        "importo": round(random.uniform(200, 5000), 2),
        "data_emissione": data_emissione.date(),
        "data_scadenza": (data_emissione + timedelta(days=30)).date()
    })

df_fatture = pd.DataFrame(fatture)
df_fatture.head(10)

,numero_fattura,cliente,importo,data_emissione,data_scadenza
0,FT-2026-001,Rossi Costruzioni SRL,3759.44,2026-01-29,2026-02-28
1,FT-2026-002,Toscana Impianti SNC,869.78,2026-03-04,2026-04-03
2,FT-2026-003,AgriToscana SOC COOP,617.31,2026-01-27,2026-02-26
3,FT-2026-004,Rossi Costruzioni SRL,343.03,2026-04-19,2026-05-19
4,FT-2026-005,Toscana Impianti SNC,2625.71,2026-02-25,2026-03-27
5,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-01-07,2026-02-06
6,FT-2026-007,Elettrodomestici Gallo SRL,1258.11,2026-05-20,2026-06-19
7,FT-2026-008,Meccanica Fiorentina SRL,4085.27,2026-05-31,2026-06-30
8,FT-2026-009,Verdi Logistica SRL,3551.07,2026-01-02,2026-02-01
9,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-03-29,2026-04-28


In [2]:
movimenti = []

def sporca_causale(numero_fattura, cliente):
    """Genera una causale realistica, scritta in uno dei tanti modi possibili"""
    n = int(numero_fattura.split("-")[2])  # es. da FT-2026-015 estrae 15
    stili = [
        f"PAGAMENTO {numero_fattura}",                  # perfetta
        f"FT {n}",                                      # abbreviata
        f"SALDO FATT. {n}/2026 {cliente.upper()}",      # con nome cliente
        f"BONIFICO DA {cliente.upper()}",               # solo nome, niente numero
        "PAGAMENTO FORNITURA",                          # inutile :)
    ]
    return random.choice(stili)

# Mescoliamo le fatture e assegniamo i destini
indici = list(df_fatture.index)
random.shuffle(indici)

non_pagate = indici[:8]          # 8 fatture restano non pagate
gruppo = indici[8:14]            # 6 fatture pagate in bonifici cumulativi
normali = indici[14:]            # il resto: pagamenti singoli

# 1) Pagamenti singoli (alcuni puliti, alcuni sporchi)
for i in normali:
    f = df_fatture.loc[i]
    ritardo = random.randint(-2, 20)  # da 2 gg in anticipo a 20 di ritardo
    importo = f["importo"]
    if random.random() < 0.15:        # 15% dei casi: commissione trattenuta
        importo = round(importo - 2.50, 2)
    movimenti.append({
        "data": f["data_scadenza"] + timedelta(days=ritardo),
        "importo": importo,
        "causale": sporca_causale(f["numero_fattura"], f["cliente"])
    })

# 2) Bonifici cumulativi: 2 bonifici da 3 fatture ciascuno
for blocco in [gruppo[:3], gruppo[3:]]:
    fatture_blocco = df_fatture.loc[blocco]
    cliente = fatture_blocco.iloc[0]["cliente"]
    movimenti.append({
        "data": fatture_blocco["data_scadenza"].max() + timedelta(days=random.randint(0, 10)),
        "importo": round(fatture_blocco["importo"].sum(), 2),
        "causale": f"SALDO FATTURE MESE {cliente.upper()}"
    })

# 3) Rumore: movimenti che non c'entrano con le fatture
rumore = [
    {"data": datetime(2026, 2, 16).date(), "importo": -1850.00, "causale": "F24 TASSE E CONTRIBUTI"},
    {"data": datetime(2026, 3, 27).date(), "importo": -4200.00, "causale": "STIPENDI FEBBRAIO"},
    {"data": datetime(2026, 3, 31).date(), "importo": -12.50, "causale": "COMMISSIONI TENUTA CONTO"},
    {"data": datetime(2026, 4, 15).date(), "importo": 500.00, "causale": "GIROCONTO INTERNO"},
]
movimenti.extend(rumore)

df_movimenti = pd.DataFrame(movimenti).sort_values("data").reset_index(drop=True)
print(f"Movimenti totali: {len(df_movimenti)}")
df_movimenti.head(10)

Movimenti totali: 52


,data,importo,causale
0,2026-02-01,3472.21,PAGAMENTO FORNITURA
1,2026-02-02,3548.57,FT 9
2,2026-02-16,-1850.00,F24 TASSE E CONTRIBUTI
3,2026-02-20,300.96,SALDO FATT. 60/2026 STUDIO NERI & ASSOCIATI
4,2026-02-22,4142.15,PAGAMENTO FORNITURA
5,2026-02-26,4116.91,PAGAMENTO FT-2026-052
6,2026-02-27,4265.47,PAGAMENTO FORNITURA
7,2026-03-02,617.31,SALDO FATT. 3/2026 AGRITOSCANA SOC COOP
8,2026-03-07,3247.90,SALDO FATT. 22/2026 INFOSERVICE SRL
9,2026-03-09,4333.29,SALDO FATT. 35/2026 ROSSI COSTRUZIONI SRL


In [3]:
df_fatture.to_csv("fatture.csv", index=False)
df_movimenti.to_csv("estratto_conto.csv", index=False)
print("File salvati!")

File salvati!


In [10]:
import re

# Estraiamo il numero progressivo da ogni fattura (es. da FT-2026-035 -> 35)
df_fatture["numero"] = df_fatture["numero_fattura"].str.split("-").str[2].astype(int)

# Dizionario: numero -> riga della fattura (per lookup veloce)
fatture_per_numero = {row["numero"]: idx for idx, row in df_fatture.iterrows()}

risultati = []

for idx, mov in df_movimenti.iterrows():
    match_trovato = None

    # 1. Estraggo TUTTI i numeri dalla causale
    numeri_in_causale = [int(n) for n in re.findall(r"\d+", mov["causale"])]

    # 2. Tengo solo quelli che corrispondono a fatture esistenti
    candidati = [n for n in numeri_in_causale if n in fatture_per_numero]

    # 3. Tra i candidati, cerco quello con importo corrispondente
    for n in candidati:
        fattura = df_fatture.loc[fatture_per_numero[n]]
        if abs(mov["importo"] - fattura["importo"]) < 0.01:
            match_trovato = fattura["numero_fattura"]
            break

    risultati.append({
        "data_movimento": mov["data"],
        "importo": mov["importo"],
        "causale": mov["causale"],
        "fattura_matchata": match_trovato,
        "livello": "1 - esatto" if match_trovato else None
    })

df_risultati = pd.DataFrame(risultati)
matchati = df_risultati["fattura_matchata"].notna().sum()
print(f"Livello 1: matchati {matchati} movimenti su {len(df_risultati)}")
df_risultati[df_risultati["fattura_matchata"].notna()].head(10)

Livello 1: matchati 23 movimenti su 52


,data_movimento,importo,causale,fattura_matchata,livello
3,2026-02-20,300.96,SALDO FATT. 60/2026 STUDIO NERI & ASSOCIATI,FT-2026-060,1 - esatto
5,2026-02-26,4116.91,PAGAMENTO FT-2026-052,FT-2026-052,1 - esatto
7,2026-03-02,617.31,SALDO FATT. 3/2026 AGRITOSCANA SOC COOP,FT-2026-003,1 - esatto
8,2026-03-07,3247.90,SALDO FATT. 22/2026 INFOSERVICE SRL,FT-2026-022,1 - esatto
9,2026-03-09,4333.29,SALDO FATT. 35/2026 ROSSI COSTRUZIONI SRL,FT-2026-035,1 - esatto
10,2026-03-20,4001.98,FT 36,FT-2026-036,1 - esatto
12,2026-03-23,2115.16,SALDO FATT. 57/2026 MECCANICA FIORENTINA SRL,FT-2026-057,1 - esatto
18,2026-04-07,869.78,PAGAMENTO FT-2026-002,FT-2026-002,1 - esatto
24,2026-04-18,4745.36,PAGAMENTO FT-2026-051,FT-2026-051,1 - esatto
28,2026-05-02,1976.87,FT 20,FT-2026-020,1 - esatto
